In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
df = pd.read_csv("../data/processed/cleaned_data.csv")

In [3]:
df["clean_text"].isnull().sum()

np.int64(1)

In [4]:
df[df["clean_text"].isnull()]

,sentiment,text,text_length,clean_text
2983,neutral,It 's not .,4,NaN


In [5]:
df = df.dropna(subset=["clean_text"])

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"], df["sentiment"],
    test_size=0.2, random_state=42, stratify=df["sentiment"]
)

In [7]:
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,1))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [8]:
print(X_train_tfidf.shape)
print(X_test_tfidf.shape)

(3876, 5000)
(969, 5000)


In [9]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

In [10]:
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)

MultinomialNB()

In [11]:
y_pred_nb = nb_model.predict(X_test_tfidf)

In [12]:
print("Accuracy:", accuracy_score(y_test, y_pred_nb))
print(classification_report(y_test, y_pred_nb))

Accuracy: 0.6873065015479877
              precision    recall  f1-score   support

    negative       1.00      0.07      0.12       121
     neutral       0.70      0.96      0.81       576
    positive       0.60      0.39      0.47       272

    accuracy                           0.69       969
   macro avg       0.77      0.47      0.47       969
weighted avg       0.71      0.69      0.63       969



In [13]:
# Logistic Regression & SVM

In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

In [15]:
lr_model = LogisticRegression(max_iter=1000)

In [16]:
lr_model.fit(X_train_tfidf, y_train)
y_pred_lr = lr_model.predict(X_test_tfidf)

In [17]:
svm_model = LinearSVC()

In [18]:
svm_model.fit(X_train_tfidf, y_train)
y_pred_svm = svm_model.predict(X_test_tfidf)

In [19]:
print("--- Logistic Regression ---")
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

--- Logistic Regression ---
Accuracy: 0.7337461300309598
              precision    recall  f1-score   support

    negative       0.71      0.43      0.54       121
     neutral       0.74      0.93      0.83       576
    positive       0.71      0.45      0.55       272

    accuracy                           0.73       969
   macro avg       0.72      0.60      0.64       969
weighted avg       0.73      0.73      0.71       969



In [20]:
print("--- SVM ---")
print("Accuracy:", accuracy_score(y_test, y_pred_svm))
print(classification_report(y_test, y_pred_svm))

--- SVM ---
Accuracy: 0.7430340557275542
              precision    recall  f1-score   support

    negative       0.66      0.57      0.61       121
     neutral       0.78      0.86      0.82       576
    positive       0.68      0.56      0.62       272

    accuracy                           0.74       969
   macro avg       0.71      0.67      0.68       969
weighted avg       0.74      0.74      0.74       969



In [21]:
results_df = pd.DataFrame({
    "text": df.loc[X_test.index, "text"],   # original, readable text
    "actual": y_test,
    "predicted": y_pred_svm
})

In [22]:
misclassified = results_df[results_df["actual"] != results_df["predicted"]]
print(f"Total misclassified: {len(misclassified)} out of {len(results_df)}")

Total misclassified: 249 out of 969


In [23]:
neg_as_neutral = misclassified[(misclassified["actual"] == "negative") & (misclassified["predicted"] == "neutral")]
neg_as_neutral[["text", "actual", "predicted"]]

,text,actual,predicted
4788,"According to the company , in addition to norm...",negative,neutral
697,Finnish power supply solutions and systems pro...,negative,neutral
4449,Stora Enso 's target has been cut to EUR 4.85 ...,negative,neutral
4320,"In a media advisory , the NTSB said that after...",negative,neutral
1986,The SeaWind that was en route from the Finnish...,negative,neutral
4372,`` Those uncertainties cloud the long-term out...,negative,neutral
4075,The poorest index figure was given to Finnish ...,negative,neutral
3590,Finnish sports equipment company Amer Sports s...,negative,neutral
4621,Finnish construction company YIT is reducing t...,negative,neutral
4151,"According to Swedish authorities , traces of t...",negative,neutral


In [24]:
pd.set_option('display.max_colwidth', None)
neg_as_neutral[["text", "actual", "predicted"]].head(15)

,text,actual,predicted
4788,"According to the company , in addition to normal seasonal fluctuation the market situation has weakened during autumn 2008 .",negative,neutral
697,"Finnish power supply solutions and systems provider Efore Oyj said its net loss widened to 3.2 mln euro $ 4.2 mln for the first quarter of fiscal 2006-2007 ending October 31 , 2007 from 900,000 euro $ 1.2 mln for the same period of fiscal 2005-06 .",negative,neutral
4449,Stora Enso 's target has been cut to EUR 4.85 from EUR 5.55 and Holmen 's target -- to SEK 135 from SEK 150 .,negative,neutral
4320,"In a media advisory , the NTSB said that after subsequent testing , `` the train detection system intermittently failed . ''",negative,neutral
1986,The SeaWind that was en route from the Finnish port of Turku to Stockholm got stuck in ice already around 4 p.m. on Wednesday and the Regal Star 's journey from the Swedish port of Kapellskar to Paldiski in northwestern Estonia was cut short at 2 a.m. on Thursday .,negative,neutral
4372,`` Those uncertainties cloud the long-term outlook . '',negative,neutral
4075,"The poorest index figure was given to Finnish power company Fortum , 4.5 .",negative,neutral
3590,Finnish sports equipment company Amer Sports said it has decided to lay off 370 workers from its Salomon division in France .,negative,neutral
4621,"Finnish construction company YIT is reducing the number of start-ups of market-financed residential units in 2007 to about 2,300 from the previously announced 2,700 .",negative,neutral
4151,"According to Swedish authorities , traces of the very toxic osmium tetroxide have been found on the coast of Per+Æmeri , the Northernmost part of the Gulf of Bothnia .",negative,neutral


In [25]:
import numpy as np

In [26]:
feature_names = np.array(vectorizer.get_feature_names_out())
classes = nb_model.classes_

In [27]:
for i, cls in enumerate(classes):
    top_indices = np.argsort(nb_model.feature_log_prob_[i])[-15:][::-1]
    print(f"\n--- Top words for '{cls}' ---")
    print(feature_names[top_indices])


--- Top words for 'negative' ---
['eur' 'mn' 'profit' 'operating' 'period' 'year' 'decreased' 'sale' 'loss'
 'quarter' 'compared' 'net' '2009' '2008' 'million']

--- Top words for 'neutral' ---
['company' 'share' 'eur' 'finland' 'million' 'service' 'sale' 'business'
 'market' 'said' 'group' 'new' 'value' 'finnish' 'also']

--- Top words for 'positive' ---
['eur' 'mn' 'year' 'profit' 'sale' 'company' 'net' 'said' 'period'
 'million' 'operating' 'rose' 'finnish' 'mln' 'increased']
